# BeamFinder — Detection Notebook

Runs the fine-tuned YOLO26s model on test images and exports detections to CSV.

**Prerequisite:** Run `train.ipynb` first (or have `best.pt` weights available).

In [ ]:
# Install dependencies (kernel may restart — this is normal, just continue to the next cell)
!pip install -q ultralytics

In [ ]:
# Clone repo and cd into it
import os

if not os.path.exists("beamfinder"):
    !git clone https://github.com/seriouslysahid/beamfinder.git
else:
    !cd beamfinder && git pull

%cd beamfinder

In [ ]:
# Configuration
from pathlib import Path

MODEL = "runs/drone_detect/weights/best.pt"  # or load from Drive: "/content/drive/MyDrive/beamfinder_weights/best.pt"
IMAGE_DIR = Path("data/images/test")
OUTPUT_DIR = Path("output")
CONF = 0.4
IMGSZ = 960

In [ ]:
# Run inference and save detections to CSV
import csv
from ultralytics import YOLO

model = YOLO(MODEL)
csv_path = OUTPUT_DIR / "detections.csv"
annotated_dir = OUTPUT_DIR / "annotated"
csv_path.parent.mkdir(parents=True, exist_ok=True)
annotated_dir.mkdir(parents=True, exist_ok=True)

total = 0

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["image", "x_center", "y_center", "width", "height", "confidence", "class"])

    results = model.predict(
        source=str(IMAGE_DIR), conf=CONF, imgsz=IMGSZ,
        save=True, project=str(OUTPUT_DIR), name="annotated",
        exist_ok=True,
    )
    for r in results:
        name = Path(r.path).name
        if r.boxes is not None and len(r.boxes):
            for box in r.boxes:
                cx, cy, w, h = box.xywh[0].tolist()
                writer.writerow([name, round(cx, 2), round(cy, 2),
                                 round(w, 2), round(h, 2),
                                 round(box.conf.item(), 4),
                                 r.names[int(box.cls.item())]])
                total += 1

print(f"{total} detections saved to {csv_path}")

In [ ]:
# Show sample annotated images
from IPython.display import Image, display
import random

annotated = sorted(annotated_dir.glob("*.jpg"))
samples = random.sample(annotated, min(6, len(annotated)))
for img in samples:
    display(Image(filename=str(img), width=600))

In [ ]:
# Preview CSV
import pandas as pd

df = pd.read_csv(csv_path)
print(f"Total detections: {len(df)}")
df.head(10)

In [ ]:
# Download results (optional)
from google.colab import files

!zip -q -r output.zip output/
files.download("output.zip")